In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification, BertForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
from torch.optim import Adam, AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm
import os
import json


In [42]:
def pretrain_global_model(model_name, train_dataset, test_dataset, batch_size=32, num_epochs=30, learning_rate=1e-5, 
                        optimizer_type='adamw', use_gpu=True, verbose=True, output_dir=None):
    """Pretrains the global model on the training dataset before federated learning.
    
    Args:
        model: The transformer model to be trained
        train_dataset: The dataset for pretraining
        batch_size: Batch size for training
        num_epochs: Number of pretraining epochs
        learning_rate: Learning rate for optimizer
        optimizer_type: Type of optimizer ('adam', 'adamw')
        use_gpu: Whether to use GPU if available
        verbose: Whether to display progress information
        output_dir: Directory to save the final model (None = don't save)
        
    Returns:
        The pretrained model
    """
    # Create a simple args object for tokenize_dataset since it expects args
    dataset = 'sst2' if 'sentence' in train_dataset.column_names else 'ag_news'
    
    
    
    text_field_key = 'text' if dataset == 'ag_news' else 'sentence'
    if model_name == 'bert':
        tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
    elif model_name == 'distilbert':
        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
    tokenized_dataset = train_dataset.map(lambda x: tokenizer(x[text_field_key], padding=True, truncation=True, max_length=512), batched=True)
    tokenized_dataset = tokenized_dataset.with_format('torch')
    
    # Create data loader
    trainloader = DataLoader(
        tokenized_dataset, batch_size=batch_size, shuffle=True)
    
    # Set device
    if use_gpu:
        device = 'cuda' if torch.cuda.is_available() else 'mps'
    else:
        device = 'cpu'
        
    print("Training model on device:", device)
    
    model.to(device)
    model.train()
    
    # Setup optimizer
    if optimizer_type.lower() == 'adam':
        optimizer = Adam(model.parameters(), lr=learning_rate)
    elif optimizer_type.lower() == 'adamw':
        optimizer = AdamW(model.parameters(), lr=learning_rate)
    else:
        # Default to AdamW
        optimizer = AdamW(model.parameters(), lr=learning_rate)
    
    # Loss function
    criterion = CrossEntropyLoss()
    
    verbose = True
    
    # Training loop
    for epoch in range(num_epochs):
        batch_losses = []
        # Add progress bar for batches
        pbar = tqdm(trainloader, 
                   desc=f'Pretraining | Epoch: {epoch+1}',
                   leave=False, 
                   disable=not verbose)
        
        for batch_idx, batch in enumerate(pbar):
            # Move data to device
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(inputs, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Track loss
            batch_losses.append(loss.item())
            
            # Update progress bar with current loss
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        
        # Calculate average epoch loss
        epoch_loss = sum(batch_losses) / len(batch_losses) if batch_losses else 0
        
        print(f'| Pretraining | Epoch: {epoch+1} | Average Loss: {epoch_loss:.4f}')
    
    # Evaluate the model after pretraining
    if test_dataset is not None:
        model.eval()
        
        test_dataset = test_dataset.map(lambda x: tokenizer(x[text_field_key], padding=True, truncation=True, max_length=512), batched=True)
        test_dataset = test_dataset.with_format('torch')
        testloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        acc, loss = 0, 0
        for batch in testloader:
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(inputs, attention_mask=attention_mask)
            logits = outputs.logits
            loss += criterion(logits, labels).item()
            preds = torch.argmax(logits, dim=1)
            acc += (preds == labels).sum().item()
            
        acc /= len(test_dataset)
        loss /= len(test_dataset)
        
        print(f'| Pretraining Complete | Accuracy: {acc:.4f} | Loss: {loss:.4f}')
    
    output_dir = f'save/pretrained_model_{model_name}'
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)

In [45]:
model_name = "bert"
    
def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data
# load dataset
train_path = 'data/sst2_train.jsonl'
test_path = 'data/sst2_test.jsonl'
train_dataset = load_jsonl(train_path)[:1000]
test_dataset = load_jsonl(test_path)

train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

if isinstance(train_dataset, dict):
    train_dataset = Dataset.from_dict(train_dataset)
if isinstance(test_dataset, dict):
    test_dataset = Dataset.from_dict(test_dataset)

In [46]:
pretrain_global_model(model_name, train_dataset, test_dataset)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training model on device: mps


| Pretraining | Epoch: 1 | Average Loss: 0.6679


| Pretraining | Epoch: 2 | Average Loss: 0.4952


...

| Pretraining | Epoch: 8 | Average Loss: 0.0459


| Pretraining | Epoch: 9 | Average Loss: 0.0208


| Pretraining | Epoch: 10 | Average Loss: 0.0152


Map:   0%|          | 0/872 [00:00<?, ? examples/s]

| Pretraining Complete | Accuracy: 0.8578 | Loss: 0.0169
